# 08. 베이스라인: Whisper + 키워드 규칙 + 상태머신 — **Colab GPU**

```
음성 → Whisper large-v3 (단어별 시각) → 키워드 규칙으로 '일' 추출 → 상태머신 → 질의 시각별 B-S-O → 평가
```

- 일부러 **단순한** 기준선. 규칙은 템플릿 목록에 맞추지 않고 일반적인 중계 표현 수준으로 만든다
- 판정을 잘못 세도 카운트 언급("투앤원")·타자 소개("원아웃, 타석에는")가 나오면 상태머신이 바로잡는다

**준비**: Drive `kbo-listen/dataset_pilot/` (06 결과), Colab `/content/kbolisten/` 업로드 (state_machine.py, evaluate.py 포함 최신본)

In [1]:
!pip install -q jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.7 MB/s eta 0:00:00


In [8]:
import json
import re
import time
from pathlib import Path

import jiwer
import torch
from google.colab import drive
from transformers import pipeline

from kbolisten.evaluate import evaluate, load_groups
from kbolisten.state_machine import answer_queries

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/kbo-listen")
VERSION = "v2"  # 평가할 파일럿 버전 (v1: 판정 생략 없음 / v2: Hard에 판정 생략)
DATA = DRIVE / ("dataset_pilot" if VERSION == "v1" else f"dataset_pilot_{VERSION}")
OUT = DRIVE / ("baseline_rules" if VERSION == "v1" else f"baseline_rules_{VERSION}")
OUT.mkdir(exist_ok=True)

queries = json.loads((DATA / "queries.json").read_text(encoding="utf-8"))
gt = json.loads((DATA / "gt.json").read_text(encoding="utf-8"))
groups = load_groups(DATA / "meta.jsonl")
print(DATA.name, len(queries), "클립", sum(len(ts) for ts in queries.values()), "질의")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset_pilot_v2 20 클립 309 질의


In [9]:
# 1) Whisper로 받아쓰기 (단어별 시작·끝 시각)
asr = pipeline(
    "automatic-speech-recognition", model="openai/whisper-large-v3",
    torch_dtype=torch.float16, device="cuda:0",
)

transcripts = {}
t0 = time.perf_counter()
for cid in queries:
    out = asr(
        str(DATA / "audio" / f"{cid}.wav"),
        return_timestamps="word", chunk_length_s=30, batch_size=2,
        generate_kwargs={"language": "korean", "task": "transcribe"},
    )
    words = []
    for ch in out["chunks"]:
        start, end = ch["timestamp"]
        words.append({"text": ch["text"], "start": start, "end": end if end is not None else start})
    transcripts[cid] = {"text": out["text"], "words": words}
print(f"받아쓰기 {time.perf_counter() - t0:.0f}초")

(OUT / "transcripts.json").write_text(json.dumps(transcripts, ensure_ascii=False, indent=1), encoding="utf-8")
print(transcripts[next(iter(transcripts))]["text"][:300])

Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcripti

받아쓰기 575초
 8회 말, 기아 공격입니다. 스코어 롯데 5대 기아 5, 마운드에는 김상수, 노아웃, 타석에는 이우성, 슬라이더, 먹혔고 파울입니다. 자, 왼쪽 파울입니다. 볼카운트 노볼투 스트라이크, 3구 129킬로 포크, 볼입니다. 볼카운트 원앤투, 129킬로 포크, 파울이 됐고요. 원볼 투 스트라이크입니다. 자, 높게 빠집니다. 높은 공 골라냅니다. 볼카운트 풀카운트 슬라이더 뜬 공 잡았습니다. 원아웃 타석에는 김태군 초구 골라냅니다. 볼카운트 원볼 일단 빠집니다. 자, 골이고요. 볼카운트 쓰리 볼 143킬로 직구, 낮은 스트라이크 3볼 1


In [10]:
# 2) 받아쓰기 품질: 실제 대본과 비교한 CER (글자 오류율). 띄어쓰기·문장부호는 빼고 비교
def norm(text):
    return re.sub(r"[\s.,!?]", "", text)


scripts = {}
with (DATA / "scripts.jsonl").open(encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        scripts[s["clip_id"]] = " ".join(u["text"] for u in s["utts"])

cer = jiwer.cer([norm(scripts[c]) for c in queries], [norm(transcripts[c]["text"]) for c in queries])
print(f"CER {cer:.2%}")

CER 5.03%


In [11]:
# 3) 키워드 규칙: 위에서부터 우선. 앞 규칙에 걸린 글자는 뒤 규칙이 다시 쓰지 않는다
#    (예: '볼카운트 투앤원'의 '볼'을 볼 판정으로 세지 않도록 카운트 규칙이 먼저 가져감)
NUM = {"노": 0, "원": 1, "투": 2, "쓰리": 3, "쓰": 3, "0": 0, "1": 1, "2": 2, "3": 3}
N = r"(노|원|투|쓰리|쓰|[0-3])"

RULES = [
    # 중계에서 직접 들은 상태 → 보정
    (r"풀\s?카운트", lambda m: ("set_count", {"b": 3, "s": 2})),
    (N + r"\s?(?:앤|-)\s?" + N, lambda m: ("set_count", {"b": NUM[m[1]], "s": NUM[m[2]]})),
    (N + r"\s?볼\s?" + N + r"\s?스트라이크", lambda m: ("set_count", {"b": NUM[m[1]], "s": NUM[m[2]]})),
    (r"(원|투|쓰리|[1-3])\s?볼(?!넷)", lambda m: ("set_count", {"b": NUM[m[1]], "s": 0})),  # '투볼' = 2-0 관례
    (N + r"\s?아웃", lambda m: ("set_outs", {"o": NUM[m[1]]})),
    (r"타석에", lambda m: ("new_batter", {})),
    # 타석 종료
    (r"(?:헛\s?스윙|루킹|스트라이크)?[,\s]*(?:삼진|3진)", lambda m: ("in_play", {"outs": 1})),
    (r"볼넷|고의\s?(?:4구|사구)|몸에\s?맞|사구|걸어\s?나", lambda m: ("walk", {})),
    (r"삼중살|트리플\s?플레이", lambda m: ("in_play", {"outs": 3})),
    (r"병살|더블\s?플레이", lambda m: ("in_play", {"outs": 2})),
    (r"안타|[23]루타|홈런|실책", lambda m: ("in_play", {"outs": 0})),
    (r"(?:잡아냅니다|잡았습니다|처리합니다)?[,.\s]*아웃|잡아냅니다|잡았습니다|처리합니다|희생|야수\s?선택",
     lambda m: ("in_play", {"outs": 1})),
    # 투구 판정
    (r"파울|커트", lambda m: ("foul", {})),
    (r"헛\s?스윙|스트라이크", lambda m: ("strike", {})),
    (r"(?<![땅뜬])볼(?!넷|\s?카운트)|빠집니다|빠졌|골라", lambda m: ("ball", {})),
]
RULES = [(re.compile(p), f) for p, f in RULES]


def extract_actions(words):
    """단어 목록 → [(시각, 메서드, 인자, 걸린 글자)]. 시각은 걸린 글자가 끝나는 단어의 끝 시각"""
    text, word_end_at = "", []  # word_end_at[i] = 글자 i가 속한 단어의 끝 시각
    for w in words:
        text += w["text"]
        word_end_at += [w["end"]] * len(w["text"])

    used = [False] * len(text)
    actions = []
    for pattern, make in RULES:
        for m in pattern.finditer(text):
            if any(used[m.start():m.end()]):
                continue
            used[m.start():m.end()] = [True] * (m.end() - m.start())
            name, kwargs = make(m)
            actions.append((word_end_at[m.end() - 1], name, kwargs, m.group(0)))
    return sorted(actions, key=lambda a: a[0])


# 첫 클립에서 뽑힌 일 몇 개 확인
cid = next(iter(transcripts))
for t, name, kwargs, hit in extract_actions(transcripts[cid]["words"])[:15]:
    print(f"{t:6.2f}  {name:<10} {kwargs}  ← '{hit.strip()}'")

  7.46  set_outs   {'o': 0}  ← '노아웃'
  8.08  new_batter {}  ← '타석에'
 11.40  foul       {}  ← '파울'
 13.92  foul       {}  ← '파울'
 15.78  set_count  {'b': 0, 's': 2}  ← '노볼투 스트라이크'
 18.68  ball       {}  ← '볼'
 20.32  set_count  {'b': 1, 's': 2}  ← '원앤투'
 23.14  foul       {}  ← '파울'
 25.34  set_count  {'b': 1, 's': 2}  ← '원볼 투 스트라이크'
 27.40  ball       {}  ← '빠집니다'
 29.92  ball       {}  ← '골라'
 31.00  set_count  {'b': 3, 's': 2}  ← '풀카운트'
 34.16  in_play    {'outs': 1}  ← '잡았습니다'
 34.58  set_outs   {'o': 1}  ← '원아웃'
 35.10  new_batter {}  ← '타석에'


In [12]:
# 4) 예측 + 평가
pred = {}
for cid, ts in queries.items():
    actions = [(t, name, kwargs) for t, name, kwargs, _ in extract_actions(transcripts[cid]["words"])]
    pred[cid] = answer_queries(actions, ts)

(OUT / "pred.json").write_text(json.dumps(pred, ensure_ascii=False), encoding="utf-8")
scores = evaluate(gt, pred, groups)
(OUT / "score.json").write_text(json.dumps(scores, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"{'group':<12} {'n':>5} {'exact':>7} {'ball':>7} {'strike':>7} {'out':>7}")
for name, s in scores.items():
    print(f"{name:<12} {s['n']:>5} {s['exact']:>7.2%} {s['b']:>7.2%} {s['s']:>7.2%} {s['o']:>7.2%}")

group            n   exact    ball  strike     out
all            309  87.70%  90.94%  93.20%  98.38%
dev             59  76.27%  83.05%  88.14%  96.61%
dev/easy        31  87.10%  90.32%  90.32%  96.77%
dev/hard        28  64.29%  75.00%  85.71%  96.43%
test           149  90.60%  93.96%  94.63% 100.00%
test/easy       80  91.25%  93.75%  96.25% 100.00%
test/hard       69  89.86%  94.20%  92.75% 100.00%
train          101  90.10%  91.09%  94.06%  97.03%
train/easy      47  95.74%  95.74%  95.74%  95.74%
train/hard      54  85.19%  87.04%  92.59%  98.15%


In [13]:
# 5) 오답 분석: 틀린 질의 바로 앞에 무슨 말이 있었고 무엇을 뽑았는지
def show_errors(cid, limit=8):
    words = transcripts[cid]["words"]
    actions = extract_actions(words)
    shown = 0
    for g, p in zip(gt[cid], pred[cid]):
        if (g["b"], g["s"], g["o"]) == (p["b"], p["s"], p["o"]):
            continue
        heard = "".join(w["text"] for w in words if g["t"] - 4 <= w["end"] <= g["t"])
        took = [f"{n}{kw or ''}" for t, n, kw, _ in actions if g["t"] - 4 <= t <= g["t"]]
        print(f"t={g['t']:6.2f} 정답 {g['b']}-{g['s']} {g['o']}아웃 / 예측 {p['b']}-{p['s']} {p['o']}아웃")
        print(f"         들은 말: {heard.strip()}")
        print(f"         뽑은 일: {took}")
        shown += 1
        if shown == limit:
            break


worst = min(queries, key=lambda c: evaluate({c: gt[c]}, pred)["all"]["exact"])
print("가장 많이 틀린 클립:", worst, groups[worst])
show_errors(worst)

가장 많이 틀린 클립: 20251004SSHT02025_06B ['dev', 'dev/hard']
t= 15.81 정답 0-0 0아웃 / 예측 1-0 0아웃
         들은 말: 낮게 빠집니다, 직구
         뽑은 일: ['ball']
t= 26.28 정답 0-2 0아웃 / 예측 0-1 0아웃
         들은 말: 노볼원 스트라이크입니다, 뒤로 가는
         뽑은 일: ["set_count{'b': 0, 's': 1}"]
t= 36.85 정답 2-2 1아웃 / 예측 0-0 1아웃
         들은 말: 타석에는 박찬호, 카운트 투 앤
         뽑은 일: ['new_batter']
